In [1]:
from collections import defaultdict
from itertools import combinations


# ============================================================
# 1. FP-TREE NODE
# ============================================================

class FPNode:
    def __init__(self, item, count=1, parent=None):
        self.item = item
        self.count = count
        self.parent = parent

        # Child pointers
        self.children = {}

        # Header-table node link
        self.node_link = None

    def add_child(self, item):
        """Create a child node."""
        child = FPNode(item, 1, self)
        self.children[item] = child
        return child


# ============================================================
# 2. FP-GROWTH CLASS
# ============================================================

class FPGrowth:

    def __init__(self, transactions, min_support=2):

        self.transactions = transactions
        self.min_support = min_support

        # Frequent item support counts
        self.item_support = defaultdict(int)

        # Header table:
        # item -> [support, first_node]
        self.header_table = {}

        # Root of FP-tree
        self.root = FPNode(None, 0, None)

        # Store frequent itemsets
        self.frequent_itemsets = {}

    # ========================================================
    # 3. FIND FREQUENT ITEMS
    # ========================================================

    def find_frequent_items(self):

        for transaction in self.transactions:
            for item in transaction:
                self.item_support[item] += 1

        # Remove infrequent items
        self.item_support = {
            item: count
            for item, count in self.item_support.items()
            if count >= self.min_support
        }

    # ========================================================
    # 4. INSERT NODE INTO HEADER TABLE
    # ========================================================

    def update_header(self, item, node):

        if item not in self.header_table:
            self.header_table[item] = [
                self.item_support[item],
                node
            ]
            return

        current = self.header_table[item][1]

        # Follow node links
        while current.node_link is not None:
            current = current.node_link

        current.node_link = node

    # ========================================================
    # 5. INSERT TRANSACTION INTO FP-TREE
    # ========================================================

    def insert_transaction(self, transaction, node, count=1):

        if not transaction:
            return

        item = transaction[0]

        # Existing child
        if item in node.children:

            child = node.children[item]
            child.count += count

        # New child
        else:

            child = FPNode(
                item,
                count,
                node
            )

            node.children[item] = child

            self.update_header(item, child)

        # Recursively insert remaining items
        self.insert_transaction(
            transaction[1:],
            child,
            count
        )

    # ========================================================
    # 6. BUILD FP-TREE
    # ========================================================

    def build_tree(self):

        self.find_frequent_items()

        for transaction in self.transactions:

            # Keep only frequent items
            filtered = [
                item
                for item in transaction
                if item in self.item_support
            ]

            # Sort by descending support
            filtered.sort(
                key=lambda x: self.item_support[x],
                reverse=True
            )

            if filtered:
                self.insert_transaction(
                    filtered,
                    self.root
                )

    # ========================================================
    # 7. DISPLAY TREE RECURSIVELY
    # ========================================================

    def print_tree(self, node=None, level=0):

        if node is None:
            node = self.root

        for child in node.children.values():

            print(
                "  " * level +
                f"{child.item}:{child.count}"
            )

            self.print_tree(
                child,
                level + 1
            )

    # ========================================================
    # 8. FIND PREFIX PATH
    # ========================================================

    def get_prefix_path(self, node):

        path = []

        parent = node.parent

        while parent is not None and parent.item is not None:

            path.append(parent.item)

            parent = parent.parent

        path.reverse()

        return path

    # ========================================================
    # 9. CONDITIONAL PATTERN BASE
    # ========================================================

    def find_conditional_pattern_base(self, item):

        patterns = []

        node = self.header_table[item][1]

        # Follow header-table node links
        while node is not None:

            path = self.get_prefix_path(node)

            if path:
                patterns.append(
                    (path, node.count)
                )

            node = node.node_link

        return patterns

    # ========================================================
    # 10. BUILD CONDITIONAL FP-TREE
    # ========================================================

    def build_conditional_tree(self, pattern_base):

        support = defaultdict(int)

        # Calculate support of items
        for path, count in pattern_base:

            for item in path:
                support[item] += count

        # Remove infrequent items
        support = {
            item: count
            for item, count in support.items()
            if count >= self.min_support
        }

        root = FPNode(None, 0, None)

        header = {}

        def update_header(item, node):

            if item not in header:

                header[item] = [
                    support[item],
                    node
                ]

            else:

                current = header[item][1]

                while current.node_link:
                    current = current.node_link

                current.node_link = node

        def insert(path, node, count):

            if not path:
                return

            item = path[0]

            if item in node.children:

                child = node.children[item]
                child.count += count

            else:

                child = FPNode(
                    item,
                    count,
                    node
                )

                node.children[item] = child

                update_header(
                    item,
                    child
                )

            insert(
                path[1:],
                child,
                count
            )

        # Insert every conditional path
        for path, count in pattern_base:

            filtered = [
                item
                for item in path
                if item in support
            ]

            filtered.sort(
                key=lambda x: support[x],
                reverse=True
            )

            if filtered:

                insert(
                    filtered,
                    root,
                    count
                )

        return root, header

    # ========================================================
    # 11. RECURSIVE FP-GROWTH
    # ========================================================

    def mine_tree(
        self,
        tree,
        header,
        prefix
    ):

        # Sort items by ascending support
        items = sorted(
            header.keys(),
            key=lambda x: header[x][0]
        )

        for item in items:

            new_pattern = prefix + [item]

            support = header[item][0]

            # Store frequent itemset
            self.frequent_itemsets[
                frozenset(new_pattern)
            ] = support

            # ----------------------------------------------
            # Construct conditional pattern base
            # ----------------------------------------------

            pattern_base = []

            node = header[item][1]

            while node is not None:

                path = []

                parent = node.parent

                while parent is not None and parent.item is not None:

                    path.append(parent.item)

                    parent = parent.parent

                path.reverse()

                if path:

                    pattern_base.append(
                        (path, node.count)
                    )

                node = node.node_link

            # ----------------------------------------------
            # Build conditional FP-tree
            # ----------------------------------------------

            if pattern_base:

                conditional_root, conditional_header = \
                    self.build_conditional_tree(
                        pattern_base
                    )

                # ------------------------------------------
                # Recursive mining
                # ------------------------------------------

                if conditional_header:

                    self.mine_tree(
                        conditional_root,
                        conditional_header,
                        new_pattern
                    )

    # ========================================================
    # 12. RUN FP-GROWTH
    # ========================================================

    def fit(self):

        self.build_tree()

        self.mine_tree(
            self.root,
            self.header_table,
            []
        )

        return self.frequent_itemsets

    # ========================================================
    # 13. GENERATE ASSOCIATION RULES
    # ========================================================

    def generate_rules(self, min_confidence=0.6):

        rules = []

        # Support lookup
        support = self.frequent_itemsets

        for itemset in support:

            if len(itemset) < 2:
                continue

            itemset_support = support[itemset]

            # Generate all possible antecedents
            for size in range(
                1,
                len(itemset)
            ):

                for antecedent in combinations(
                    itemset,
                    size
                ):

                    antecedent = frozenset(
                        antecedent
                    )

                    consequent = itemset - antecedent

                    if antecedent in support:

                        confidence = (
                            itemset_support /
                            support[antecedent]
                        )

                        if confidence >= min_confidence:

                            rules.append({
                                "antecedent":
                                    set(antecedent),

                                "consequent":
                                    set(consequent),

                                "support":
                                    itemset_support,

                                "confidence":
                                    confidence
                            })

        return rules


# ============================================================
# 14. SAMPLE DATASET
# ============================================================

transactions = [

    ["Milk", "Bread", "Eggs"],

    ["Milk", "Bread"],

    ["Milk", "Diaper", "Beer", "Bread"],

    ["Milk", "Diaper", "Beer", "Cola"],

    ["Milk", "Diaper", "Bread"],

    ["Bread", "Eggs"],

    ["Milk", "Bread", "Diaper", "Beer"],

    ["Bread", "Milk", "Eggs"]

]


# ============================================================
# 15. TRAIN FP-GROWTH
# ============================================================

fp = FPGrowth(
    transactions,
    min_support=3
)

frequent_itemsets = fp.fit()


# ============================================================
# 16. DISPLAY FREQUENT ITEMSETS
# ============================================================

print("FREQUENT ITEMSETS")
print("=" * 50)

for itemset, support in frequent_itemsets.items():

    print(
        set(itemset),
        "Support:",
        support
    )


# ============================================================
# 17. DISPLAY FP-TREE
# ============================================================

print("\nFP-TREE")
print("=" * 50)

fp.print_tree()


# ============================================================
# 18. GENERATE ASSOCIATION RULES
# ============================================================

rules = fp.generate_rules(
    min_confidence=0.6
)

print("\nASSOCIATION RULES")
print("=" * 50)

for rule in rules:

    print(
        rule["antecedent"],
        "->",
        rule["consequent"],
        "| Support:",
        rule["support"],
        "| Confidence:",
        round(rule["confidence"], 2)
    )

FREQUENT ITEMSETS
{'Eggs'} Support: 3
{'Eggs', 'Bread'} Support: 3
{'Beer'} Support: 3
{'Beer', 'Milk'} Support: 3
{'Beer', 'Diaper'} Support: 3
{'Beer', 'Diaper', 'Milk'} Support: 3
{'Diaper'} Support: 4
{'Diaper', 'Bread'} Support: 3
{'Diaper', 'Milk', 'Bread'} Support: 3
{'Diaper', 'Milk'} Support: 4
{'Milk'} Support: 7
{'Bread'} Support: 7
{'Milk', 'Bread'} Support: 5

FP-TREE
Milk:6
  Bread:5
    Eggs:1
    Diaper:3
      Beer:2
  Diaper:1
    Beer:1
Bread:2
  Eggs:1
  Milk:1
    Eggs:1

ASSOCIATION RULES
{'Eggs'} -> {'Bread'} | Support: 3 | Confidence: 1.0
{'Beer'} -> {'Milk'} | Support: 3 | Confidence: 1.0
{'Beer'} -> {'Diaper'} | Support: 3 | Confidence: 1.0
{'Diaper'} -> {'Beer'} | Support: 3 | Confidence: 0.75
{'Beer'} -> {'Diaper', 'Milk'} | Support: 3 | Confidence: 1.0
{'Diaper'} -> {'Beer', 'Milk'} | Support: 3 | Confidence: 0.75
{'Beer', 'Diaper'} -> {'Milk'} | Support: 3 | Confidence: 1.0
{'Beer', 'Milk'} -> {'Diaper'} | Support: 3 | Confidence: 1.0
{'Diaper', 'Milk'} ->